In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
import joblib
import time

In [2]:
df = pd.read_csv("../../datasets/training_dataset/train.csv")

In [3]:
X = df[["is_https", "cte_domain", "subdomains", "domain_length", 
        "hyphens", "dots", "uppercase", "se_domain"]]
y = df['label']

In [5]:
print("\nTraining Decision Tree...")
start = time.time()
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X, y)
y_pred_dt = dt.predict(X)
end = time.time()
print(f"Decision Tree Training Time: {end - start:.2f} sec")


Training Decision Tree...
Decision Tree Training Time: 16.91 sec


In [13]:
print("\nDecision Tree Report:")
print("Accuracy:", accuracy_score(y, y_pred_dt))
print(classification_report(y, y_pred_dt))
joblib.dump(dt, "./models_after_pm/decision_tree_model_with_no_feature_filtration.pkl")


Decision Tree Report:
Accuracy: 0.88632875
              precision    recall  f1-score   support

           0       0.89      0.90      0.90   2696949
           1       0.88      0.86      0.87   2103051

    accuracy                           0.89   4800000
   macro avg       0.89      0.88      0.88   4800000
weighted avg       0.89      0.89      0.89   4800000



['./models_after_pm/decision_tree_model_with_no_feature_filtration.pkl']

In [7]:
train_data = lgb.Dataset(X, label=y)

params = {
    "objective": "binary",  
    "boosting": "gbdt",
    "metric": "binary_error", 
    "num_leaves": 64,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

print("\nTraining LightGBM on full train.csv ...")
start = time.time()

# Train model (no validation split)
model = lgb.train(
    params,
    train_data,
    num_boost_round=500
)

end = time.time()
print(f"✅ LightGBM Training Time: {end - start:.2f} sec")

# Predictions on training data
y_pred = model.predict(X)
y_pred_binary = (y_pred > 0.5).astype(int)


Training LightGBM on full train.csv ...
✅ LightGBM Training Time: 43.23 sec


In [14]:
print("\nLightGBM Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred_binary))
print(classification_report(y, y_pred_binary))
model.save_model("./models_after_pm/lightgbm_model.txt")



LightGBM Report (Train Set):
Accuracy: 0.8980520833333333
              precision    recall  f1-score   support

           0       0.92      0.90      0.91   2696949
           1       0.88      0.89      0.88   2103051

    accuracy                           0.90   4800000
   macro avg       0.90      0.90      0.90   4800000
weighted avg       0.90      0.90      0.90   4800000



In [9]:
nb = GaussianNB()
nb.fit(X, y)

y_pred = nb.predict(X)

print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))

Accuracy: 0.7313822916666667
              precision    recall  f1-score   support

           0       0.68      0.98      0.80   2696949
           1       0.93      0.42      0.58   2103051

    accuracy                           0.73   4800000
   macro avg       0.81      0.70      0.69   4800000
weighted avg       0.79      0.73      0.70   4800000



In [10]:
print("\nTraining XGBoost (baseline params)...")
start = time.time()

xgb = XGBClassifier(
    n_estimators=200,       # number of trees (baseline)
    max_depth=3,            # depth of trees
    objective='binary:logistic',  # binary classification
    n_jobs=-1,
    random_state=42
)

xgb.fit(X, y)

end = time.time()
print(f"XGBoost Training Time: {end - start:.2f} sec")

# Predictions on train set
y_pred = xgb.predict(X)


Training XGBoost (baseline params)...
XGBoost Training Time: 8.54 sec


In [15]:
print("\nXGBoost Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))
xgb.save_model("./models_after_pm/xgboost_model_with_no_feature_filtration.json")


XGBoost Report (Train Set):
Accuracy: 0.88890875
              precision    recall  f1-score   support

           0       0.91      0.90      0.90   2696949
           1       0.87      0.88      0.87   2103051

    accuracy                           0.89   4800000
   macro avg       0.89      0.89      0.89   4800000
weighted avg       0.89      0.89      0.89   4800000



In [16]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nTraining MLP (baseline params)...")
start = time.time()

# Baseline MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),  
    activation='relu',            
    solver='adam',                
    learning_rate_init=0.001,      
    max_iter=20,                 
    random_state=42,
    verbose=True                   
)

mlp.fit(X_scaled, y)

end = time.time()
print(f"✅ MLP Training Time: {end - start:.2f} sec")
y_pred = mlp.predict(X_scaled)


Training MLP (baseline params)...
Iteration 1, loss = 0.25758618
Iteration 2, loss = 0.24357292
Iteration 3, loss = 0.24132585
Iteration 4, loss = 0.24007962
Iteration 5, loss = 0.23918986
Iteration 6, loss = 0.23850927
Iteration 7, loss = 0.23798098
Iteration 8, loss = 0.23755599
Iteration 9, loss = 0.23718920
Iteration 10, loss = 0.23685645
Iteration 11, loss = 0.23653104
Iteration 12, loss = 0.23628156
Iteration 13, loss = 0.23610468
Iteration 14, loss = 0.23590614
Iteration 15, loss = 0.23577696
Iteration 16, loss = 0.23560450
Iteration 17, loss = 0.23542750
Iteration 18, loss = 0.23529301
Iteration 19, loss = 0.23513746
Iteration 20, loss = 0.23509937
✅ MLP Training Time: 2586.64 sec


C:\Users\soman\AppData\Roaming\Python\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [17]:
print("\nMLP Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))


MLP Report (Train Set):
Accuracy: 0.8930670833333333
              precision    recall  f1-score   support

           0       0.92      0.89      0.90   2696949
           1       0.86      0.90      0.88   2103051

    accuracy                           0.89   4800000
   macro avg       0.89      0.89      0.89   4800000
weighted avg       0.89      0.89      0.89   4800000



In [18]:
joblib.dump(mlp, "./models_after_pm/mlp_model.pkl")
joblib.dump(scaler, "./models_after_pm/mlp_scaler.pkl")

['./models_after_pm/mlp_scaler.pkl']